In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split
from sklearn.metrics import silhouette_score

import torch
import torch.nn as nn

from ortools.constraint_solver import pywrapcp, routing_enums_pb2

In [2]:
df = pd.read_csv("train.csv")

In [ ]:
df["Order_Date"] = pd.to_datetime(df["Order_Date"])
df["Time_Orderd"] = pd.to_datetime(df["Time_Orderd"], format="%H:%M:%S", errors="coerce")
df["Type_of_vehicle"] = df["Type_of_vehicle"].str.strip()
df = df[
    (df["Restaurant_latitude"] != 0.0) &
    (df["Restaurant_longitude"] != 0.0) &
    (df["Delivery_location_latitude"] != 0.0) &
    (df["Delivery_location_longitude"] != 0.0) &
    (df["Type_of_vehicle"].isin(["scooter","motorcycle"])) 
]
df.dropna(inplace=True)

In [4]:
temp_df = df[["Restaurant_latitude", "Restaurant_longitude"]]
temp_df = temp_df.groupby(by=["Restaurant_latitude", "Restaurant_longitude"]).size().reset_index(name="Count")
temp_df.sort_values(by="Count", ascending=False, inplace=True)
temp_df = temp_df[
    (temp_df["Restaurant_latitude"].apply(lambda x: 26.0 <= x <= 27.0)) &
    (temp_df["Restaurant_longitude"].apply(lambda x: 75.0 <= x <= 76.0))
]

In [5]:
start_time = pd.to_datetime("00:01:01").time()
end_time = pd.to_datetime("23:59:59").time()
start_date = pd.to_datetime("2022-03-01")
end_date = pd.to_datetime("2022-04-06")
full_date_range = pd.date_range(start=start_date, end=end_date)
restaurant_latitude = temp_df["Restaurant_latitude"].mean()
restaurant_longitude = temp_df["Restaurant_longitude"].mean()

In [6]:
main_df = df[
    (df["Restaurant_latitude"].isin(temp_df["Restaurant_latitude"])) &
    (df["Restaurant_longitude"].isin(temp_df["Restaurant_longitude"])) &
    (df["Time_Orderd"].dt.time.between(start_time, end_time)) &
    (df["Order_Date"].between(start_date, end_date))
]

In [ ]:
scaler1 = StandardScaler()
main_df[["Scaled_Customer_Latitude", "Scaled_Customer_Longitude"]] = scaler1.fit_transform(main_df[["Delivery_location_latitude", "Delivery_location_longitude"]])

In [8]:
def find_best_k(data, max_k):
    silhouette_scores = []
    kmeans_labels = []
    for k in range(2, max_k+1):
        kmeans_optimal = KMeans(n_clusters=k)
        kmeans_optimal.fit(data)
        kmeans_labels.append((k, kmeans_optimal.labels_))
        silhouette_scores.append((k, silhouette_score(data, kmeans_optimal.labels_)))
    return silhouette_scores, kmeans_labels

In [ ]:
silhouette_scores, kmeans_labels = find_best_k(main_df[["Scaled_Customer_Latitude", "Scaled_Customer_Longitude"]], 10)
best_k, best_score = max(silhouette_scores, key=lambda x: x[1])
best_labels = next(lbl for k, lbl in kmeans_labels if k == best_k)
main_df["K-Means Label"] = best_labels

In [ ]:
plt.scatter(x=main_df["Delivery_location_latitude"], y=main_df["Delivery_location_longitude"], c=main_df["K-Means Label"])
plt.xlabel("Customer Location Latitudes")
plt.ylabel("Customer Location Longitudes")
plt.legend()
plt.show()

In [11]:
cluster_df = main_df[main_df["K-Means Label"] == 1]
cluster_df = cluster_df.groupby(by="Order_Date").size().reset_index(name="Number of Deliveries")
cluster_df.sort_values(by="Order_Date", inplace=True)
cluster_df.set_index(keys="Order_Date", inplace=True)
cluster_df = cluster_df.reindex(labels=full_date_range, fill_value=0)
cluster_df.reset_index(inplace=True)
cluster_df.columns = ["Order_Date", "Number of Deliveries"]

In [12]:
data = cluster_df["Number of Deliveries"].to_numpy()

In [13]:
def create_sequences(data, sequence_length):
    sequences = []
    for i in range(len(data)-sequence_length):
        seq = data[i:i+sequence_length]
        lbl = data[i+sequence_length]
        sequences.append((seq, lbl))
    return sequences

In [14]:
sequences = create_sequences(data, 5)

In [15]:
X = np.array([seq[0] for seq in sequences])
y = np.array([[lbl[1]] for lbl in sequences])

In [16]:
scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()

X_scaled = scaler_X.fit_transform(X)
y_scaled = scaler_y.fit_transform(y)

In [17]:
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y_scaled, test_size=0.2, random_state=42)

In [18]:
X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.float32)

In [19]:
class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        self.lstm = nn.LSTM(input_size=input_size, hidden_size=hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)
    
    def forward(self, x):
        out, _ = self.lstm(x)
        output = self.fc(out[:, -1, :])
        return output

In [20]:
model = LSTMModel(1, 2, 1)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
epochs = 100

In [ ]:
for epoch in range(epochs):
    model.train()
    output = model(X_train.unsqueeze(-1))
    loss = criterion(output, y_train)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    print(f"Epoch: {epoch+1} Loss: {loss.item()}")

In [ ]:
model.eval()
with torch.no_grad():
    predictions = model(X_test.unsqueeze(-1))
    loss = criterion(predictions, y_test)
    print(f"Test Loss: {loss.item()}")

In [ ]:
plt.plot(predictions.squeeze(-1).numpy(), label="Predictions", marker="x")
plt.plot(y_test.squeeze(-1).numpy(), label="Actual Values", marker="o")
plt.legend()
plt.show()

In [24]:
predictions_real = scaler_y.inverse_transform(predictions.numpy())
y_test_real = scaler_y.inverse_transform(y_test.numpy())

In [ ]:
plt.plot(predictions_real, label="Predictions", marker="x")
plt.plot(y_test_real, label="Actual Values", marker="o")
plt.legend()
plt.show()